In [2]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 15.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 29.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [3]:
import textwrap
from collections import Counter

import altair as alt
import pandas as pd

In [4]:
def format_percentage(row, col_name):
    count = int(row["counts"])
    percentage = row["percentage"] * 100
    w = "response"
    if count > 1:
        w += "s"
    return f"{percentage:.0f}% ({count} {w})"

In [5]:
def add_formatted_column(data):
    total = data["counts"].sum()
    data["percentage"] = data["counts"] / total
    data["formatted"] = data.apply(format_percentage, axis=1, col_name="Time")
    return data

In [6]:
def wrap(text, width):
    return textwrap.wrap(text, width)

In [7]:
def alt_text(title, options, counts, order):
    text = []
    text.append(f"{title}")
    if order == "-x":
        pairs = zip(options, counts)
        sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
        for o, c in sorted_pairs:
            text.append(f"- {o}: {c}")
    else:
        d = {o: c for o, c in zip(options, counts)}
        for o in order:
            if o in d:
                text.append(f"- {o}: {d[o]}")
    print("\n".join(text))
    with open("alt-text.md", "a") as f:
        f.write("\n\n")
        f.write("\n".join(text))

In [8]:
def plot(data, title, order, output_file, subtitle=[]):
    c = Counter(data[question].dropna().tolist())
    options, counts = zip(*c.items())

    alt_text(title, options, counts, order)

    data_question = pd.DataFrame(
        {
            "options": options,
            "counts": counts,
        }
    )

    _data = add_formatted_column(data_question)

    bars = (
        alt.Chart(
            _data,
            title=alt.Title(
                wrap(title, width=40),
                subtitle=subtitle,
            ),
        )
        .mark_bar()
        .encode(
            x=alt.X("percentage:Q", axis=alt.Axis(format=".0%", title=None)),
            y=alt.Y("options:N", axis=alt.Axis(title=None, labels=True), sort=order),
            color=alt.Color("options:N", legend=None),
        )
    )

    text = (
        alt.Chart(_data)
        .mark_text(align="left", baseline="middle", dx=3)
        .encode(
            x=alt.X("percentage:Q"),
            y=alt.Y("options:N", sort=order),
            text=alt.Text("formatted"),
        )
    )

    chart = bars + text

    # chart = chart.properties(width=600, height=250)

    chart = chart.configure_title(
        fontSize=15,
        subtitleFontSize=15,
        anchor="start",
        orient="bottom",
        offset=15,
        subtitleColor="gray",
    )

    chart.show()
    chart.save(output_file, scale_factor=2)

In [12]:
data = pd.read_csv("data.csv", sep=";")
data

,"In your estimate, how much time per month have you saved as a result of attending a CodeRefinery workshop","After attending the workshop, would you judge your code to be more reusable or not more reusable?","After attending the workshop, has it become easier or not for you to collaborate on software development with your colleagues and collaborators?",Have you introduced one or more of your colleagues to new tools or practices as a result of the workshop?,How likely is it that you would recommend CodeRefinery workshop to a friend or colleague?,Has anything else changed in how you write code for your research after attending the workshop?,What would be your preferred delivery style for this workshop?,What would be your preferred delivery style for this workshop? Open text answers,What topic(s) are missing from our workshops?,What topic(s) were least interesting/helpful?,Anything else you would like to tell us about the workshop format?,Participation style,Participation style Open text answers,Career stage,Career stage Open text answers,Academic discipline,Academic discipline Open text answers
0,Days,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,10.0,NaN,In-person teaching with exercises,NaN,NaN,NaN,NaN,Learner in a team (with your colleagues/friends),NaN,Graduate student / PhD student,NaN,"Electrical Engineering, Electronic Engineering...",NaN
1,Hours,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,10.0,NaN,Live taught online lectures with demos,NaN,NaN,NaN,NaN,Learner in a team (with your colleagues/friends),NaN,Graduate student / PhD student,NaN,"Electrical Engineering, Electronic Engineering...",NaN
2,Minutes,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,10.0,"yes, I revisited my git skills, learned testin...",Live taught online lectures with exercises,NaN,I can not think but API development maybe!,NaN,Workshop lectures at first week were a bit int...,Individual learner,NaN,Graduate student / PhD student,NaN,Computer and Information Sciences,NaN
3,Hours,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,10.0,NaN,Live taught online lectures with demos,NaN,NaN,NaN,NaN,Individual learner,NaN,Researcher,NaN,Earth and Related Environmental Sciences,NaN
4,Minutes,Not sure,Collaboration is easier,Not sure,8.0,NaN,Live taught online lectures with demos,NaN,NaN,NaN,NaN,Individual learner,NaN,Graduate student / PhD student,NaN,Other Natural Sciences,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,NaN,Not sure,Not sure,Not sure,8.0,NaN,Live taught online lectures with exercises,NaN,NaN,NaN,NaN,Individual learner,NaN,Undergraduate student,NaN,Physical Sciences,NaN
68,Hours,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,9.0,Now I use AI to write some parts of the code o...,Live taught online lectures with exercises,NaN,How to best work with modern AI tools. (which ...,NaN,"I think the format was excellent already, no n...",Learner in a team (with your colleagues/friends),NaN,Graduate student / PhD student,NaN,Other:,Pharmacy
69,Minutes,Not sure,Collaboration is easier,I have not introduced one or more of my collea...,7.0,NaN,In-person teaching with exercises,NaN,NaN,NaN,NaN,Individual learner,NaN,Graduate student / PhD student,NaN,Mathematics,NaN
70,No time saved,My code is more reusable,Collaboration is easier,I have introduced one or more of my colleagues...,9.0,I helped me to think more detailed about the w...,Live taught online lectures with demos,NaN,NaN,deploy in the cloud,I really like the practical part. But I would ...,Individual learner,NaN,Graduate student / PhD student,NaN,Computer and Information Sciences,NaN


In [13]:
question = "In your estimate, how much time per month have you saved as a result of attending a CodeRefinery workshop"
order = ["No time saved", "Minutes", "Hours", "Days"]

plot(data, question, order, "time-saved.png")

In your estimate, how much time per month have you saved as a result of attending a CodeRefinery workshop
- No time saved: 11
- Minutes: 9
- Hours: 37
- Days: 13


alt.LayerChart(...)

In [14]:
question = "After attending the workshop, would you judge your code to be more reusable or not more reusable?"
order = ["My code is more reusable", "My code is not more reusable", "Not sure"]

plot(data, question, order, "reusable.png")

After attending the workshop, would you judge your code to be more reusable or not more reusable?
- My code is more reusable: 55
- My code is not more reusable: 3
- Not sure: 14


alt.LayerChart(...)

In [15]:
question = "After attending the workshop, has it become easier or not for you to collaborate on software development with your colleagues and collaborators?"
order = ["Collaboration is easier", "Collaboration is not easier", "Not sure"]

plot(data, question, order, "collaboration.png")

After attending the workshop, has it become easier or not for you to collaborate on software development with your colleagues and collaborators?
- Collaboration is easier: 49
- Collaboration is not easier: 3
- Not sure: 20


alt.LayerChart(...)

In [16]:
question = "Have you introduced one or more of your colleagues to new tools or practices as a result of the workshop?"
order = [
    "I have introduced one or more of my colleagues to new tools or practices",
    "I have not introduced one or more of my colleagues to new tools or practices",
    "Not sure",
]

plot(data, question, order, "colleagues.png")

Have you introduced one or more of your colleagues to new tools or practices as a result of the workshop?
- I have introduced one or more of my colleagues to new tools or practices: 48
- I have not introduced one or more of my colleagues to new tools or practices: 19
- Not sure: 5


alt.LayerChart(...)

In [17]:
question = "How likely is it that you would recommend CodeRefinery workshop to a friend or colleague?"
order = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]

plot(
    data,
    question,
    order,
    "recommending.png",
    subtitle="0 means definitely not. 10 means definitely yes.",
)

How likely is it that you would recommend CodeRefinery workshop to a friend or colleague?
- 10: 37
- 9: 12
- 8: 13
- 7: 6
- 6: 2
- 4: 1


alt.LayerChart(...)

In [18]:
question = "What would be your preferred delivery style for this workshop?"
order = [
    "MOOC (Asynchronous learning on learning platform)",
    "Prerecorded lectures and live discussion sessions",
    "Live taught online lectures with exercises",
    "Live taught online lectures with demos",
    "In-person teaching with exercises",
    "In-person teaching with demos",
    "Local classroom for streamed workshop with exercises"
]

plot(data, question, order, "pre-recorded-or-live-or-in-person.png")

What would be your preferred delivery style for this workshop?
- MOOC (Asynchronous learning on learning platform): 8
- Prerecorded lectures and live discussion sessions: 5
- Live taught online lectures with exercises: 30
- Live taught online lectures with demos: 16
- In-person teaching with exercises: 6
- In-person teaching with demos: 2
- Local classroom for streamed workshop with exercises: 3


alt.LayerChart(...)

In [19]:
question = "Participation style"
order = [
    "Individual learner",
    "Learner in a team (with your colleagues/friends)",
    "Learner in a local classroom",
    "Other",
]

plot(data, question, order, "participation-style.png")

Participation style
- Individual learner: 58
- Learner in a team (with your colleagues/friends): 7
- Learner in a local classroom: 6


alt.LayerChart(...)

In [20]:
question = "Career stage"
order = [
    "Undergraduate student",
    "Graduate student/ PhD student",
    "Postdoc",
    "Researcher",
    "Professor",
    "Research software engineer/ Scientific programmer",
    "Industry",
    "Other",
]

plot(data, question, order, "career-stage.png")

Career stage
- Undergraduate student: 1
- Postdoc: 15
- Researcher: 10
- Professor: 2


alt.LayerChart(...)

In [21]:
question = "Academic discipline"
order = "-x"

plot(data, question, order, "academic-discipline.png")

Academic discipline
- Physical Sciences: 14
- Computer and Information Sciences: 9
- Earth and Related Environmental Sciences: 9
- Biological Sciences: 9
- Mathematics: 8
- Health Sciences: 4
- Electrical Engineering, Electronic Engineering, Information Engineering: 2
- Chemical Sciences: 2
- Languages and Literature: 2
- Other:: 2
- Psychology: 2
- Other Natural Sciences: 1
- Economics and Business: 1
- Medical Engineering: 1
- Civil Engineering: 1
- Basic Medicine: 1
- Mechanical Engineering: 1
- Other Agricultural Sciences: 1
- Environmental Engineering: 1
- Other Engineering and Technologies: 1


alt.LayerChart(...)